# Reporter mAP histogram

Per-reporter distinctiveness across genes: a boxplot of the raw per-gene distinctiveness values for each reporter, and a histogram of the per-reporter mean distinctiveness. Both reference the `all_combined` row from the same CSV (the median for the boxplot, the mean for the histogram).

## Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42

FIGURES_DIR = Path("../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV path

Raw per-gene distinctiveness for each reporter, with one extra column `all_combined` representing the all-reporters-combined embedding.

**Before public release**, replace this cell with download instructions (or a pointer to the public dataset) and update the constant to match the released layout.

In [ ]:
CSV_PATH = "/hpc/mydata/alexander.hillsley/ops/ops_monorepo/ops_model/analysis/heatmaps/gene_reporter_distinctiveness_raw.csv"

## Load

`df` is genes × reporters; `all_combined` is the all-reporters-combined column popped off so it can be plotted as a separate reference line.

In [ ]:
df = pd.read_csv(CSV_PATH, index_col="gene")
all_combined = df.pop("all_combined")
print(f"reporters: {df.shape[1]}, genes: {df.shape[0]}")
df.head()

## Per-reporter summary statistics

Mean / median / std / quartile summary per reporter, sorted by median distinctiveness (descending). The boxplot below uses this ordering.

In [ ]:
stats = pd.DataFrame({
    "mean":   df.mean(),
    "median": df.median(),
    "std":    df.std(),
    "min":    df.min(),
    "q25":    df.quantile(0.25),
    "q75":    df.quantile(0.75),
    "max":    df.max(),
}).sort_values("median", ascending=False)
stats

## Boxplot — per-reporter distinctiveness across genes

One box per reporter, ordered by median distinctiveness. Dashed red line is the median distinctiveness of the all-reporters-combined embedding.

In [ ]:
order = stats.index.tolist()
all_combined_median = all_combined.median()

fig, ax = plt.subplots(figsize=(18, 6))
ax.boxplot(
    [df[col].values for col in order],
    tick_labels=order,
    showfliers=True,
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
)
ax.axhline(
    all_combined_median,
    linestyle="--",
    color="red",
    label=f"all_combined median = {all_combined_median:.3f}",
)
ax.set_ylabel("distinctiveness")
ax.set_xlabel("reporter")
ax.set_title("Per-reporter distinctiveness across genes")
ax.tick_params(axis="x", rotation=90)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reporter_distinctiveness_boxplot.svg", bbox_inches="tight")
plt.show()

## Histogram — distribution of per-reporter mean distinctiveness

Steelblue bars are individual reporters (height = count); the crimson bar marks the all-reporters-combined embedding (height = 1, width = histogram bin width) at its mean distinctiveness, so its position in the distribution is directly comparable to the per-reporter bars.

In [ ]:
all_combined_mean = all_combined.mean()

fig, ax = plt.subplots(figsize=(5, 5))
counts, bin_edges, _ = ax.hist(stats["mean"], bins=30, color="steelblue", edgecolor="black", label="reporters")
bin_width = bin_edges[1] - bin_edges[0]

ax.bar(
    all_combined_mean,
    1,
    width=bin_width,
    color="crimson",
    edgecolor="black",
    align="center",
    label=f"all reporters combined = {all_combined_mean:.3f}",
)

ax.set_xlabel("mean distinctiveness")
ax.set_ylabel("Number of Reporters")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reporter_distinctiveness_histogram.svg", bbox_inches="tight")
plt.show()